In [1]:
import torch
import os
from datetime import datetime
import pickle
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

BATCH_SIZE = 64
LEARNING_RATE = 1e-4
EPOCHS = 5
NEG_SAMPLE_K = 4
MAX_HISTORY = 50
MAX_TITLE_LEN = 30
NUM_HEADS = 16
HEAD_DIM = 16

torch.manual_seed(21)

In [2]:
class AdditiveAttention(nn.Module):
    def __init__(self, dim, hidden_dim=200):
        super().__init__()
        self.proj = nn.Linear(dim, hidden_dim)
        self.query = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, X, mask=None):
        e = torch.tanh(self.proj(X))
        scores = self.query(e).squeeze(-1)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        weights = F.softmax(scores, dim=-1)
        return torch.bmm(weights.unsqueeze(1), X).squeeze(1)

class NewsEncoder(nn.Module):
    def __init__(self, embedding_matrix, num_heads=16, head_dim=16, dropout=0.2):
        super().__init__()
        embed_dim = embedding_matrix.shape[1]
        self.word_embed = nn.Embedding.from_pretrained(torch.FloatTensor(embedding_matrix), freeze=False)
        self.dropout = nn.Dropout(dropout)
        attn_dim = num_heads * head_dim
        self.proj = nn.Linear(embed_dim, attn_dim)
        self.multihead_attn = nn.MultiheadAttention(embed_dim=attn_dim, num_heads=num_heads, batch_first=True)
        self.additive_attn = AdditiveAttention(attn_dim)

    def forward(self, title_ids):
        X = self.dropout(self.word_embed(title_ids))
        X = self.proj(X)
        X, _ = self.multihead_attn(X, X, X)
        X = self.dropout(X)
        news_vec = self.additive_attn(X)
        return news_vec

class UserEncoder(nn.Module):
    def __init__(self, news_dim, num_heads=16, head_dim=16, dropout=0.2):
        super().__init__()
        self.multihead_attn = nn.MultiheadAttention(embed_dim=news_dim, num_heads=num_heads, batch_first=True)
        self.additive_attn = AdditiveAttention(news_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, clicked_news_vecs, mask=None):
        X, _ = self.multihead_attn(clicked_news_vecs, clicked_news_vecs, clicked_news_vecs)
        X = self.dropout(X)
        user_vec = self.additive_attn(X, mask)
        return user_vec

class NRMSModel(nn.Module):
    def __init__(self, embedding_matrix, num_heads=16, head_dim=16, dropout=0.2):
        super().__init__()
        news_dim = num_heads * head_dim
        self.news_encoder = NewsEncoder(embedding_matrix, num_heads, head_dim, dropout)
        self.user_encoder = UserEncoder(news_dim, num_heads, head_dim, dropout)

    def forward(self, history_ids, candidate_ids, hist_mask=None):
        batch, hist_len, tlen = history_ids.shape
        _, n_cand, _ = candidate_ids.shape

        hist_flat = history_ids.view(-1, tlen)
        hist_vecs = self.news_encoder(hist_flat)
        hist_vecs = hist_vecs.view(batch, hist_len, -1)

        cand_flat = candidate_ids.view(-1, tlen)
        cand_vecs = self.news_encoder(cand_flat)
        cand_vecs = cand_vecs.view(batch, n_cand, -1)

        user_vec = self.user_encoder(hist_vecs, hist_mask)

        scores = torch.bmm(cand_vecs, user_vec.unsqueeze(-1)).squeeze(-1)
        return scores

In [3]:
class MINDDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        hist_sum = s['history'].sum(axis=1)
        mask = (hist_sum > 0).astype(np.float32)

        return {
            'history': torch.tensor(s['history'], dtype=torch.long),
            'candidates': torch.tensor(s['candidates'], dtype=torch.long),
            'labels': torch.tensor(s['labels'], dtype=torch.float),
            'hist_mask': torch.tensor(mask, dtype=torch.float)
        }

In [4]:
with open("../data/processed/MINDsmall_train.pkl", "rb") as f:
    data = pickle.load(f)

train_samples = data['train_samples']
embedding_matrix = data['embedding_matrix']

train_dataset = MINDDataset(train_samples)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on {device}")

model = NRMSModel(embedding_matrix, NUM_HEADS, HEAD_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
criterion = nn.CrossEntropyLoss()

os.makedirs("../models", exist_ok=True)

model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for batch in tqdm(train_loader):
        history = batch['history'].to(device)
        candidates = batch['candidates'].to(device)
        labels = batch['labels'].to(device)
        hist_mask = batch['hist_mask'].to(device)

        optimizer.zero_grad()
        scores = model(history, candidates, hist_mask)

        target = torch.zeros(scores.size(0), dtype=torch.long).to(device)
        loss = criterion(scores, target)
        loss.backward()
        
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()

    current_lr = optimizer.param_groups[0]['lr']
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {avg_loss:.4f}, Current LR: {current_lr}")

    scheduler.step()

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': avg_loss,
    }, f"../models/nrms_epoch{epoch+1}.pt")

Training on cuda


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3693/3693 [21:13<00:00,  2.90it/s]


Epoch 1/5, Loss: 1.4713, Current LR: 0.0001


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3693/3693 [23:29<00:00,  2.62it/s]


Epoch 2/5, Loss: 1.3996, Current LR: 0.0001


...

Epoch 3/5, Loss: 1.3647, Current LR: 5e-05


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3693/3693 [21:23<00:00,  2.88it/s]


Epoch 4/5, Loss: 1.3525, Current LR: 5e-05


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3693/3693 [21:30<00:00,  2.86it/s]


Epoch 5/5, Loss: 1.3409, Current LR: 2.5e-05


In [6]:
def dcg_score(y_true, y_score, k=10):
    order = np.argsort(y_score)[::-1][:k]
    gains = np.array(y_true)[order]
    discounts = np.log2(np.arange(len(gains)) + 2)
    return np.sum(gains / discounts)

def ndcg_score(y_true, y_score, k=10):
    best = dcg_score(y_true, y_true, k)
    if best == 0: return 0.0
    return dcg_score(y_true, y_score, k) / best

def mrr_score(y_true, y_score):
    order = np.argsort(y_score)[::-1]
    y_sorted = np.array(y_true)[order]
    for i, val in enumerate(y_sorted):
        if val == 1:
            return 1.0 / (i + 1)
    return 0.0

def evaluate(model, val_loader, device):
    model.eval()
    aucs, mrrs, ndcg5s, ndcg10s = [], [], [], []

    with torch.no_grad():
        for batch in tqdm(val_loader):
            history = batch['history'].to(device)
            candidates = batch['candidates'].to(device)
            labels = batch['labels'].numpy()
            hist_mask = batch['hist_mask'].to(device)

            scores = model(history, candidates, hist_mask).cpu().numpy()

            for i in range(len(labels)):
                y_true = labels[i]
                y_score = scores[i][:len(y_true)]
                if sum(y_true) == 0 or sum(y_true) == len(y_true):
                    continue
                aucs.append(roc_auc_score(y_true, y_score))
                mrrs.append(mrr_score(y_true, y_score))
                ndcg5s.append(ndcg_score(y_true, y_score, 5))
                ndcg10s.append(ndcg_score(y_true, y_score, 10))
    auc = np.mean(aucs)
    mrr = np.mean(mrrs)
    ndcg5 = np.mean(ndcg5s)
    ndcg10 = np.mean(ndcg10s)
    print(f"AUC: {auc:.4f}")
    print(f"MRR: {mrr:.4f}")
    print(f"nDCG@5: {ndcg5:.4f}")
    print(f"nDCG@10: {ndcg10:.4f}")
    return { 'AUC': auc, 'MRR': mrr, 'nDCG@5': ndcg5, 'nDCG@10': ndcg10 }

In [7]:
with open("../data/processed/MINDsmall_val.pkl", "rb") as f:
    data = pickle.load(f)

val_samples = data['train_samples']

val_dataset = MINDDataset(val_samples)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
model = NRMSModel(embedding_matrix, NUM_HEADS, HEAD_DIM).to(device)

files = [f for f in os.listdir("../models/") if f.endswith('.pt')]

best_loss = float('inf')
best_path = None

for file in files:
    path = os.path.join("../models/", file)
    checkpoint = torch.load(path, map_location=device)

    current_loss = checkpoint.get('loss', float('inf'))
    if current_loss < best_loss:
        best_loss = current_loss
        best_path = path

if best_path:
    print(f"Evaluating {best_path} with training loss {best_loss}")
    checkpoint = torch.load(best_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
else:
    print("No valid checkpoints found")

results = evaluate(model, val_loader, device)

results['model_path'] = best_path
results['batch_size'] = BATCH_SIZE

output_dir = "../results"
os.makedirs(output_dir, exist_ok=True)

filename = f"eval_results_{datetime.now().strftime("%Y%m%d_%H%M")}.pkl"
save_path = os.path.join(output_dir, filename)

with open(save_path, 'wb') as f:
    pickle.dump(results, f)

print(f"Evaluation results saved to {save_path}")

Evaluating ../models/nrms_epoch5.pt with training loss 1.340935490107491


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1741/1741 [05:54<00:00,  4.92it/s]

AUC: 0.4776
MRR: 0.4509
nDCG@5: 0.5847
nDCG@10: 0.5847
Evaluation results saved to ../results/eval_results_20260422_1408.pkl
